## Silver Data Quality Checks

This notebook runs basic data-quality checks on key Silver tables in the medallion pipeline. It verifies that tables exist, are non-empty, and do not have columns that are entirely NULL.

### 1. Imports and Table List

Load PySpark and define the list of Silver tables to validate. Each table is referenced by its fully-qualified name (catalog.schema.table).

In [ ]:
# Databricks notebook source
# This notebook runs basic data-quality checks on key Silver tables.

from pyspark.sql import functions as F

# Silver tables in catalog capstone, schema silver (Unity Catalog).
silver_tables = [
    "capstone.silver.dim_insurance",
    "capstone.silver.dim_pharmacy",
    "capstone.silver.fact_medication_reviews",
    "capstone.silver.location_county",
    "capstone.silver.population",
]

### 2. Helper Functions

Defines `table_exists`, `check_non_empty`, and `check_no_all_null_columns` to validate each Silver table. Fails fast if a table is missing, empty, or has columns that are entirely NULL.

In [ ]:
def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name)
        return True
    except Exception:
        return False


def check_non_empty(df, table_name: str):
    row_count = df.count()
    if row_count == 0:
        raise AssertionError(f"Silver table '{table_name}' is empty.")


def check_no_all_null_columns(df, table_name: str):
    null_only_cols = []
    for col in df.columns:
        non_null_count = df.filter(F.col(col).isNotNull()).limit(1).count()
        if non_null_count == 0:
            null_only_cols.append(col)
    if null_only_cols:
        raise AssertionError(
            f"Silver table '{table_name}' has columns that are entirely NULL: {null_only_cols}"
        )

### 3. Run Checks

Iterates over each Silver table, runs the quality checks, and raises an error if any check fails. Skipped tables (not found) are reported but do not fail the notebook.

In [ ]:
failed_tables = []
skipped_tables = []

for tbl in silver_tables:
    if not table_exists(tbl):
        skipped_tables.append(tbl)
        print(f"[SKIP] Silver table does not exist (yet): {tbl}")
        continue

    print(f"[CHECK] Silver table: {tbl}")
    df = spark.table(tbl)

    try:
        check_non_empty(df, tbl)
        check_no_all_null_columns(df, tbl)
        print(f"[OK] Silver table passed checks: {tbl}")
    except AssertionError as e:
        failed_tables.append((tbl, str(e)))
        print(f"[FAIL] {e}")

if failed_tables:
    messages = [f"{tbl}: {msg}" for tbl, msg in failed_tables]
    raise AssertionError(
        "One or more Silver tables failed data-quality checks:\n" + "\n".join(messages)
    )

print("[RESULT] Silver data-quality checks completed.")
if skipped_tables:
    print(f"[INFO] Skipped tables (not found): {skipped_tables}")